# Desafio Técnico de People Analytics: Análise de Sourcing com IA

Este notebook apresenta a solução para o desafio técnico de People Analytics, focado na análise do processo de sourcing e recrutamento. A solução inclui análise de dados, visualizações e a aplicação de Inteligência Artificial para identificar padrões e otimizar as estratégias de contratação.

## 1. Configuração e Carregamento de Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Carregar os dados
df = pd.read_csv('sourcing_data.csv')

# Exibir as primeiras linhas do dataset
print('Primeiras 5 linhas do dataset:')
print(df.head())

## 2. Análise do Funil de Recrutamento

Vamos analisar as taxas de conversão em cada etapa do funil de recrutamento.

In [ ]:
# 1. Análise de Funil de Conversão
funnel_stages = {
    'Sourced': len(df),
    'First Contact': df['response_received'].sum(),
    'Screening Pass': df['screening_pass'].sum(),
    'Interview 1 Pass': df['interview1_pass'].sum(),
    'Test Taken': df['test_taken'].sum(),
    'Offer Sent': df['offer_sent'].sum(),
    'Hired': df['hired'].sum()
}

funnel_df = pd.DataFrame(list(funnel_stages.items()), columns=['Stage', 'Count'])
funnel_df['Conversion_Rate'] = funnel_df['Count'] / funnel_df['Count'].shift(1)
funnel_df['Total_Conversion'] = funnel_df['Count'] / len(df)

print('Análise do Funil de Recrutamento:')
print(funnel_df)

### Visualização do Funil de Recrutamento

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=funnel_df, x='Stage', y='Count', color='skyblue')
plt.title('Funil de Recrutamento (Volume por Etapa)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Estratégias de Sourcing e Canais de Conversão

Vamos identificar quais canais de sourcing apresentam as melhores taxas de conversão.

In [ ]:
source_conversion = df.groupby('source_channel').agg(
    total=('candidate_id', 'count'),
    hired=('hired', 'sum')
).reset_index()
source_conversion['conversion_rate'] = (source_conversion['hired'] / source_conversion['total']) * 100
source_conversion = source_conversion.sort_values('conversion_rate', ascending=False)

print('Taxa de Conversão por Canal de Sourcing:')
print(source_conversion)

### Visualização da Taxa de Conversão por Canal

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=source_conversion, x='source_channel', y='conversion_rate', palette='viridis')
plt.title('Taxa de Conversão por Canal de Sourcing (%)')
plt.ylabel('Taxa de Conversão (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Sinais de Sucesso e Uso de IA

Para identificar os sinais que indicam maior chance de um candidato ser contratado, utilizaremos um modelo de Machine Learning (Random Forest).

In [ ]:
# Preparar dados para o modelo de IA (Predição de Hired)
features = ['source_channel', 'seniority', 'years_experience', 'department', 'work_mode']
X = df[features].copy()
y = df['hired'].astype(int)

# Encoding de variáveis categóricas
le_dict = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

# Treinar um modelo simples para ver importância de features
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# Importância das features
importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Importância das Features para Contratação:')
print(importance)

### Análise de Scores vs Contratação

In [ ]:
score_cols = ['technical_test_score', 'behavior_score', 'manager_score']
scores_hired = df.groupby('hired')[score_cols].mean().reset_index()

print('Scores Médios por Status de Contratação:')
print(scores_hired)

## 5. Recomendações Práticas para Recrutadores

1. Foco em Experiência.
2. Redução de 'No Response'.
3. Alinhamento de Expectativa Salarial.
4. Persistência.